# 01.5 — Modules and imports

Where MATLAB has the path, Python has packages. This notebook covers the entire surface area:

* `import` and its variants.
* Packages, modules, sub-modules.
* The dotted-path syntax (`from .data.dataset import ...`).
* `__init__.py` and the package boundary.
* Import errors and how to debug them.

**Prerequisite:** [01.1 syntax basics](01.1_syntax_basics.ipynb).

## Section 1 — What MATLAB does

MATLAB has a single global namespace per workspace. The MATLAB path tells the interpreter where to find `.m` files. Add a directory with `addpath`:

```matlab
addpath(genpath('/home/gerritcg/My_MATLAB'));
% Now every .m file under that directory is callable from any other .m file.
result = cgg_runAutoEncoder(1, 'SLURMChoice', 10, 'SLURMIDX', 1);
```

There is no concept of "which module did `cgg_runAutoEncoder` come from?" — it's just on the path. Two files with the same name shadow each other; the path order determines the winner.

Python is the opposite. Every name lives in some module's namespace, and you must explicitly `import` it to use it.

## Section 2 — The Python concepts you need

### 2.1 — `import module` — the simplest form

In [ ]:
import math

# Access names from the module via the dot.
print(math.pi)
print(math.sqrt(16))

You can alias a module to a shorter name with `as`. **`numpy as np`** and **`pandas as pd`** are universal conventions; don't invent your own.

In [ ]:
import numpy as np

arr = np.arange(10)
print(arr)

### 2.2 — `from module import name` — pulling specific names

This brings a name directly into your current namespace, without the module prefix.

In [ ]:
from math import sqrt, pi

print(sqrt(16))    # no module. prefix needed
print(pi)

**When to use which:**

| Style | When | Example |
|---|---|---|
| `import module` | one or two uses, or the prefix adds clarity | `import os; os.cwd()` |
| `import module as alias` | the canonical short name exists | `import numpy as np` |
| `from module import name` | you use the name often, prefix is noise | `from pathlib import Path` |
| `from module import *` | almost never — pollutes namespace | (avoid) |

The codebase uses `from pathlib import Path` everywhere because `pathlib.Path(...)` everywhere would be ugly. `numpy` and `torch` get aliased because that's the universal convention.

### 2.3 — Packages and sub-modules

A **package** is a directory with an `__init__.py` file (even an empty one). A **module** is a single `.py` file inside that directory. Packages can contain sub-packages.

Our `src/neural_data_decoding/` package looks like:

```
neural_data_decoding/
    __init__.py
    cli.py
    data/
        __init__.py
        dataset.py
        mat_dataset.py
        ...
    models/
        __init__.py
        registry.py
        ...
    sweeps/
        __init__.py
        dispatcher.py
        ...
```

Each directory with `__init__.py` is a package. Sub-packages are accessed with the dot:

In [ ]:
# Sub-module import — explicit path through the package tree
from neural_data_decoding.data.mat_dataset import MatFileTrialDataset
from neural_data_decoding.sweeps import lookup

print(MatFileTrialDataset.__module__)   # "neural_data_decoding.data.mat_dataset"
print(lookup.__module__)                 # "neural_data_decoding.sweeps.dispatcher"
                                          # (re-exported from sweeps/__init__.py)

### 2.4 — Relative imports (within a package)

When a module wants to import from a sibling in the same package, it uses a **relative import** with leading dots. **One dot** means "the current package"; **two dots** means "the parent package."

Inside `src/neural_data_decoding/cli.py`:

```python
from .data.dataset import SyntheticTrialDataset      # current package's data sub-package
from .interop import build_matlab_run_dirs            # current package's interop sub-package
from .sweeps.cli_helpers import apply_overrides       # current package's sweeps sub-package
```

Inside `src/neural_data_decoding/sweeps/cli_helpers.py`:

```python
from neural_data_decoding.sweeps.dispatcher import lookup as _lookup_sweep
                # absolute import — same effect, more explicit
```

**Rules of thumb:**
- Inside the package's own code: relative imports are fine and idiomatic.
- From outside (notebooks, tests, scripts): always use absolute imports (`from neural_data_decoding.x.y import z`).
- A dot before a name (`.data`) only works INSIDE a package's `__init__.py` or modules. Notebooks aren't part of the package, so they use absolute paths.

### 2.5 — `__init__.py` — the package's curated facade

`__init__.py` runs when the package is first imported. It controls what's exposed at the package level vs the module level.

`src/neural_data_decoding/sweeps/__init__.py`:

```python
from neural_data_decoding.sweeps.dispatcher import SWEEP_ENTRIES, SweepEntry, lookup
from neural_data_decoding.sweeps.banner import collect_banner_data, render_banner
...
__all__ = ["SWEEP_ENTRIES", "SweepEntry", "lookup", ...]
```

This lets users write either:

```python
from neural_data_decoding.sweeps import lookup            # short — re-exported
from neural_data_decoding.sweeps.dispatcher import lookup  # long — direct path
```

Both work; the first is preferred because it survives internal refactors. If we move `lookup` from `dispatcher.py` to `lookup.py` tomorrow, the package-level import stays valid; the direct one breaks.

`__all__` is a list of names that `from package import *` would pull in. The codebase always defines it because pyright uses it to decide what's public.

## Section 3 — The `neural_data_decoding` implementation

Look at the package facade for the data sub-package:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/__init__.py").read_text()
print(src)

Just five lines. Three imports from internal modules, one `__all__` list that exposes them at the package level.

Now compare to a module that USES this facade — the CLI's imports:

In [ ]:
src = Path("../../src/neural_data_decoding/cli.py").read_text().splitlines()
# Find the first import block
import_lines = []
for line in src:
    if line.startswith(("import ", "from ", "    ")) and line.strip():
        import_lines.append(line)
    elif import_lines and line.strip() == "":
        if len(import_lines) > 50:
            break
        import_lines.append("")
    elif import_lines and not line.startswith(("import ", "from ", "    ", "#")):
        break
for line in import_lines[:30]:
    print(line)

Notice three patterns:

* `from __future__ import annotations` — first; enables forward references in type hints.
* Standard library imports — `argparse`, `json`, `sys`, `pathlib`, `typing`.
* Third-party imports — `torch`, `omegaconf`.
* Project-relative imports with leading dots — `from .data.dataset import ...`, `from .interop import ...`, `from .sweeps.cli_helpers import ...`.

This grouping (stdlib, third-party, project) with blank lines between is the convention (codified in PEP 8 / enforced by `isort`).

## Section 4 — Hands-on exercises

### Exercise 4.1 — three import styles for the same name

Try all three ways to import the `lookup` function, then explain which line you'd write in your own code.

In [ ]:
# Style A — import the module, then call with the prefix
import neural_data_decoding.sweeps.dispatcher
print(neural_data_decoding.sweeps.dispatcher.lookup(1).description)

In [ ]:
# Style B — alias
import neural_data_decoding.sweeps.dispatcher as disp
print(disp.lookup(1).description)

In [ ]:
# Style C — direct via the package facade
from neural_data_decoding.sweeps import lookup
print(lookup(1).description)

Style C is what production code uses. Style A is verbose; style B is fine when you call many names from the same module but want to keep the namespace visible.

### Exercise 4.2 — fix an import error

The cell below errors. Fix it without changing anything outside this cell.

In [ ]:
# Your fix:
# from neural_data_decoding.data import SyntheticDataset  # ← what's wrong?

# Hint: ask the package what names it exposes.
from neural_data_decoding import data
print(dir(data))

In [ ]:
# Reveal:
# It's "SyntheticTrialDataset", not "SyntheticDataset" — the imported facade lists the available names.
from neural_data_decoding.data import SyntheticTrialDataset
print(SyntheticTrialDataset)

## Section 5 — Common errors

### `ModuleNotFoundError: No module named 'foo'`

The most common Python error in any setup. Three causes in order of likelihood:

1. **Wrong kernel / venv** — see [00.2 set up your environment](../00_orientation/00.2_set_up_your_environment.ipynb) for the diagnostic (`import sys; print(sys.executable)`).
2. **Forgot `pip install -e .`** — for editable installs of the local package.
3. **Typo in the module name** — `numpy` not `numpie`.

### `ImportError: cannot import name 'foo' from 'bar'`

The module exists; the name doesn't. Run `dir(bar)` to see what's actually exposed. Usually a renamed function (`build_result_dir` was replaced by `build_matlab_run_dirs` — same symptom).

### `ImportError: attempted relative import with no known parent package`

You used `from .foo import bar` in a top-level script (not inside a package). Either:
- Run as a module (`python -m neural_data_decoding.cli` rather than `python src/neural_data_decoding/cli.py`).
- Change to an absolute import.

This is the most common surprise when porting MATLAB scripts — Python's package system is more strict.

### `RecursionError` during import

You have a **circular import** — module A imports from B, and B imports from A. Refactor so the import is one-directional, or move the shared code into a third module both can import.

### Imports work in JupyterLab but not VS Code (or vice versa)

The two editors picked different Python interpreters as their kernel. See [00.2 Section 5](../00_orientation/00.2_set_up_your_environment.ipynb).

## Section 6 — Further reading

- [Python tutorial — Modules](https://docs.python.org/3/tutorial/modules.html). The canonical reference, including packages.
- [PEP 328 — relative imports](https://peps.python.org/pep-0328/). Explains the dot syntax.
- [PEP 8 — import order](https://peps.python.org/pep-0008/#imports). The stdlib / third-party / project grouping.
- [`isort`](https://pycqa.github.io/isort/) — the tool that auto-sorts your imports.

Next up: **[01.6 error handling](01.6_error_handling.ipynb)**.